# Lean 4 Theorem Proving — Progressive-Context Induction Eval

Evaluates LLM ability to complete Lean 4 tactic proofs from LeanDojo
Benchmark 4 as a function of how much context is supplied about the proof
state and the true next tactic's premises. Two cumulative rung ladders:
`stepk:0..2` (proof-state detail -- goal, hypotheses, tactics-so-far -- with
no answer-conditional content) and `hint:0..4` (premise detail -- names,
signatures, bodies, then 1-/2-hop transitive-dependency closure -- which
progressively leaks information about the true next tactic). A `noise:N`
control arm token-matches `hint:N` with lorem-ipsum filler at the same
volume, isolating *real content* from *raw context length*. Generation runs
against Prime Intellect / OpenRouter / AWS Bedrock / self-hosted-EC2 models
via `smolbench.lean.runner.sweep`; each candidate proof is verified by
replaying it through a LeanDojo `Dojo` session and checking it closes the
goal. See `notebooks/lean/README.md` for the full experimental design
(rung semantics, trivial-rung skip, dataset, pre-flight replay filter).

**NOTE:** this notebook's kernel MUST be `.venv-lean` (Python 3.12), not the
main `smolbench` (3.14) kernel. The imports below work on either venv (the
runner loads `smolbench.lean.verify` lazily, at `sweep()` call time), but
*executing* a sweep verifies every candidate proof through `lean_dojo`,
which pins `Requires-Python <3.13` upstream and is only installed in
`.venv-lean`.


In [ ]:
from pathlib import Path

import smolbench
from dotenv import load_dotenv

# Anchored via smolbench.__file__ (not Path.cwd()) so this resolves
# correctly regardless of the kernel's working directory -- notebook kernels
# commonly run with a temp-dir cwd. Mirrors notebooks/chromatic's
# RESULTS_DIR / EC2_STATE_FILE anchoring pattern (see that notebook's setup
# cell), applied here to keys.env itself rather than just to results/state
# paths, since this notebook's keys.env lives at a fixed repo-relative
# location regardless of what invoked the kernel.
EXPERIMENT_DIR = Path(smolbench.__file__).resolve().parents[1] / "notebooks" / "lean"
load_dotenv(EXPERIMENT_DIR / "keys.env")

from smolbench.lean.runner import results_root, sweep, write_run_analysis


In [ ]:
# Migrated from lean/configs/*.yaml (the four still-live configs; the retired
# single-model noise_iso variants and old main/main_v2 configs live in git
# history only -- see notebooks/lean/README.md). Schema migrations applied
# uniformly to all four, relative to the original YAML:
#   - provider: "openai_compat" -> provider: "primeintellect" (Prime
#     Intellect's OpenAI-compatible passthrough; smolbench's provider stack
#     names it "primeintellect", not the old generic "openai_compat" label)
#   - added top-level "seed" (base decoding seed), "request_timeout", and
#     "max_retries" -- fields the old YAML schema didn't have, now read by
#     smolbench.lean.runner.sweep(). Every other key/value is unchanged.

# Smoke sweep — full implemented rung ladder (8 rungs).
# 30 theorems × 8 rungs × 2 models × 1 rollout = 480 cells (minus trivials).
# With concurrent gen + Dojo session reuse: ~10-15 min wall-clock; ~$2-4.
#
# hint:2 now slices real proof bodies (not just corpus signature).
# hint:3 / hint:4 add 1-hop / 2-hop file-imports transitive closure, token-capped.
# Trivial rungs (e.g. stepk:2 at k=0; hint:* with no recorded premises) are
# auto-skipped to keep per-rung pass rates apples-to-apples.
SMOKE_CONFIG = {
    "run_name": "smoke_v3",

    # Base decoding seed; the runner derives each rollout's actual request
    # seed as `seed + rollout_idx`, so every cell in every sweep is
    # independently reproducible.
    "seed": 1776,
    # Per-request read-timeout, in seconds.
    "request_timeout": 1800,
    # Per-request cap on retryable-failure attempts (rate limits, transient
    # network errors) before a cell gives up and records an "exception" verdict.
    "max_retries": 4,

    "models": [
        # anthropic/* ids are served by Prime Intellect's passthrough here;
        # provider: "openrouter" is the alternative route for these two.
        {"provider": "primeintellect", "model": "anthropic/claude-haiku-4.5"},
        {"provider": "primeintellect", "model": "anthropic/claude-sonnet-4.6"},
    ],

    "theorems": {
        "source": "replay_passing",
        "kind": "random",
        "split": "val",
        "limit": 30,
        "max_tactics": 5,
        "seed": 0,
    },

    "k": {"strategy": "last"},

    "rungs": [
        "stepk:0", "stepk:1", "stepk:2",
        "hint:0", "hint:1", "hint:2", "hint:3", "hint:4",
    ],

    "n_rollouts": 1,
    "temperature": 0.7,
    "max_tokens": 4096,
    "dojo_timeout": 300,

    "concurrent_gen": True,
    "max_concurrency": 16,
    "skip_trivial": True,
}


# Main experiment v3 — cross-lab reasoning-toggle pollution sweep on PI-hosted
# open-weight models (no Anthropic, no OpenAI; Sonnet & GPT-5.5 dropped to fit
# $200 budget; Qwen pair dropped due to upstream qwen-instruct 429/503 quota
# instability on this PI account). 100 theorems × 10 rungs × 7 model configs × 3 rollouts.
#
# Design (3 toggle pairs across 3 labs + 1 always-on reasoning anchor):
#
#   DeepSeek MoE (671B/37B):
#     - v3.2-speciale     reasoning-on, separate fine-tune (always on)
#     - v3.2-high         reasoning-on toggle
#     - v3.2-none         reasoning-off toggle
#   Google dense (Flash tier):
#     - gemini-3-flash high     reasoning-on toggle
#     - gemini-3-flash none     reasoning-off toggle
#   Moonshot MoE (1T/32B):
#     - kimi-k2.6 high          reasoning-on toggle
#     - kimi-k2.6 none          reasoning-off toggle
#
# Toggle behavior verified 2026-05-07 via results/runs/probe_v3/probe.py:
# every "high"/"thinking" config emits substantive reasoning_content; every
# "none"/"instruct" emits zero. PI exposes reasoning_content reliably on these
# open-weight routes (unlike its OpenAI passthrough).
#
# Rung ladder (10 rungs):
#   stepk:2  no premise leak (full state + tactics-so-far + theorem identity)
#   hint:0   + premise NAMES used in true next tactic
#   hint:1   + premise SIGNATURES                  ── content step 1
#   noise:1  hint:0 padded to hint:1 token count   ── volume control 1
#   hint:2   + premise BODIES (full source)        ── content step 2
#   noise:2  hint:1 padded to hint:2 token count   ── volume control 2
#   hint:3   + 1-hop per-premise dep closure       ── content step 3
#   noise:3  hint:2 padded to hint:3 token count   ── volume control 3
#   hint:4   + 2-hop per-premise dep closure       ── content step 4
#   noise:4  hint:3 padded to hint:4 token count   ── volume control 4
#
# Cost projection at 100×10×7×3 = 21,000 cells (minus trivials, ~19k effective):
# ~$135, with ~$65 buffer.
MAIN_V3_CONFIG = {
    "run_name": "main_v3",

    "seed": 1776,
    "request_timeout": 1800,
    "max_retries": 4,

    "models": [
        {"provider": "primeintellect", "model": "deepseek/deepseek-v3.2-speciale",
         "display_name": "v3.2-speciale"},
        {"provider": "primeintellect", "model": "deepseek/deepseek-v3.2",
         "display_name": "v3.2-high", "extra_params": {"reasoning_effort": "high"}},
        {"provider": "primeintellect", "model": "deepseek/deepseek-v3.2",
         "display_name": "v3.2-none", "extra_params": {"reasoning_effort": "none"}},
        {"provider": "primeintellect", "model": "google/gemini-3-flash-preview",
         "display_name": "gemini-flash-high", "extra_params": {"reasoning_effort": "high"}},
        {"provider": "primeintellect", "model": "google/gemini-3-flash-preview",
         "display_name": "gemini-flash-none", "extra_params": {"reasoning_effort": "none"}},
        {"provider": "primeintellect", "model": "moonshotai/kimi-k2.6",
         "display_name": "kimi-k2.6-high", "extra_params": {"reasoning_effort": "high"}},
        {"provider": "primeintellect", "model": "moonshotai/kimi-k2.6",
         "display_name": "kimi-k2.6-none", "extra_params": {"reasoning_effort": "none"}},
    ],

    "theorems": {
        "source": "replay_passing",
        "kind": "random",
        "split": "val",
        "limit": 100,
        "max_tactics": 5,
        "seed": 0,
    },

    "k": {"strategy": "last"},

    "rungs": [
        "stepk:2",
        "hint:0", "hint:1", "noise:1",
        "hint:2", "noise:2",
        "hint:3", "noise:3",
        "hint:4", "noise:4",
    ],

    "n_rollouts": 3,
    "temperature": 0.7,
    "max_tokens": 32768,
    "dojo_timeout": 300,

    "concurrent_gen": True,
    "max_concurrency": 16,
    "skip_trivial": True,
    "theorem_workers": 8,
}


# Main experiment v3.2 — frontier closed-weight complement to main_v3.
#
# main_v3 covers cross-lab open-weight models (DeepSeek, Google Flash, Moonshot)
# at low cost (~$135). This sweep adds the frontier closed-weight tier that
# was dropped from main_v3 to fit budget:
#
#   Anthropic dense (Sonnet 4.6):
#     - sonnet-4.6 off       no extended thinking
#     - sonnet-4.6 thinking  extended thinking on (toggle pair)
#   OpenAI dense (GPT-5.5):
#     - gpt-5.5 high         reasoning_effort=high
#     - gpt-5.5 none         reasoning_effort=none (toggle pair)
#
# Routes via PI passthrough (same auth + plumbing as main_v3). Note: PI's
# OpenAI-passthrough only reliably exposes reasoning_content ~17% of the time
# (see main_v2 audit) — so for gpt-5.5-high we get the toggle effect on
# pass rates but NOT the CoT text for qualitative analysis. Anthropic's
# extended-thinking exposure on PI is untested; verify via probe before launch.
#
# Cost projection at 100×10×4×3 = 12,000 cells (minus trivials):
#   sonnet-4.6 off:           ~$36
#   sonnet-4.6 thinking-on:   ~$213   <-- dominates
#   gpt-5.5 high:             ~$189
#   gpt-5.5 none:             ~$57
#                            ------
#   total                    ~$495
#
# This is well over the original $200 budget. Options:
#   - run only the cheapest pair (sonnet-off + gpt-5.5-none)  ~$93
#   - drop sonnet-thinking (the most expensive single config)  ~$282
#   - reduce theorems 100→30 for this sweep                     ~$148
# Adjust before launching.
MAIN_V3_2_CONFIG = {
    "run_name": "main_v3_2",

    "seed": 1776,
    "request_timeout": 1800,
    "max_retries": 4,

    "models": [
        # anthropic/* is served by Prime Intellect's passthrough here;
        # provider: "openrouter" is the alternative route for these two.
        {"provider": "primeintellect", "model": "anthropic/claude-sonnet-4.6",
         "display_name": "sonnet-4.6-none"},
        {"provider": "primeintellect", "model": "anthropic/claude-sonnet-4.6",
         "display_name": "sonnet-4.6-thinking", "extra_params": {"reasoning_effort": "high"}},
        {"provider": "primeintellect", "model": "openai/gpt-5.5",
         "display_name": "gpt-5.5-high", "extra_params": {"reasoning_effort": "high"}},
        {"provider": "primeintellect", "model": "openai/gpt-5.5",
         "display_name": "gpt-5.5-none", "extra_params": {"reasoning_effort": "none"}},
    ],

    "theorems": {
        "source": "replay_passing",
        "kind": "random",
        "split": "val",
        "limit": 30,
        "max_tactics": 5,
        # 30-theorem subset of main_v3's set (deterministic via seed) for
        # direct cross-model comparison.
        "seed": 0,
    },

    "k": {"strategy": "last"},

    "rungs": [
        "stepk:2",
        "hint:0", "hint:1", "noise:1",
        "hint:2", "noise:2",
        "hint:3", "noise:3",
        "hint:4", "noise:4",
    ],

    "n_rollouts": 3,
    "temperature": 0.7,
    "max_tokens": 32768,
    "dojo_timeout": 300,

    "concurrent_gen": True,
    "max_concurrency": 16,
    "skip_trivial": True,
    # Share machine with main_v3 (which uses 8); stay under memory pressure.
    "theorem_workers": 2,
}


# Three-model isolation across the (MoE × reasoning) grid:
#   - MoE no-reasoning:    mistralai/mixtral-8x22b-instruct  (~141B/39B active)
#   - MoE reasoning:       nvidia/nemotron-3-super-120b-a12b (~120B/12B active)
#   - Dense no-reasoning:  meta-llama/llama-3.3-70b-instruct (dense 70B)
#
# Same 30 theorems / 5 rungs / 1 rollout shape as the other noise_iso runs.
NOISE_ISO_3WAY_CONFIG = {
    "run_name": "noise_iso_3way_v1",

    "seed": 1776,
    "request_timeout": 1800,
    "max_retries": 4,

    "models": [
        {"provider": "primeintellect", "model": "mistralai/mixtral-8x22b-instruct"},
        {"provider": "primeintellect", "model": "nvidia/nemotron-3-super-120b-a12b"},
        {"provider": "primeintellect", "model": "meta-llama/llama-3.3-70b-instruct"},
    ],

    "theorems": {
        "source": "replay_passing",
        "kind": "random",
        "split": "val",
        "limit": 30,
        "max_tactics": 5,
        # same theorem set as the other noise_iso runs
        "seed": 0,
    },

    "k": {"strategy": "last"},

    "rungs": ["hint:2", "hint:3", "noise:3", "hint:4", "noise:4"],

    "n_rollouts": 1,
    "temperature": 0.7,
    # generous for the reasoning model
    "max_tokens": 8192,
    "dojo_timeout": 300,

    "concurrent_gen": True,
    "max_concurrency": 8,
    "skip_trivial": True,
    "theorem_workers": 4,
}


## Smoke

Fast sanity sweep across the full 8-rung ladder (`stepk:0..2`, `hint:0..4`)
on 30 theorems × 2 models (Claude Haiku 4.5, Claude Sonnet 4.6) × 1 rollout
-- roughly 480 cells before trivial-skip, ~10-15 min wall-clock with
concurrent generation and Dojo session reuse, ~$2-4. Run this first after
any pipeline change, before launching a full sweep.


In [ ]:
RUN_DIR = results_root() / "runs" / SMOKE_CONFIG["run_name"]
sweep(SMOKE_CONFIG, RUN_DIR)


In [ ]:
write_run_analysis(RUN_DIR)
print((RUN_DIR / "analysis.txt").read_text())


## Main v3

Cross-lab reasoning-toggle pollution sweep on Prime-Intellect-hosted
open-weight models -- DeepSeek V3.2 (speciale/high/none), Gemini 3 Flash
(high/none), Kimi K2.6 (high/none) -- 100 theorems × 10 rungs (the full
`hint`/`noise` ladder) × 7 model configs × 3 rollouts, ~21,000 cells before
trivial-skip, cost-projected at ~$135.


In [ ]:
RUN_DIR = results_root() / "runs" / MAIN_V3_CONFIG["run_name"]
sweep(MAIN_V3_CONFIG, RUN_DIR)


In [ ]:
write_run_analysis(RUN_DIR)
print((RUN_DIR / "analysis.txt").read_text())


## Main v3.2

Frontier closed-weight complement to main v3 -- Anthropic Sonnet 4.6
(off/thinking) and OpenAI GPT-5.5 (high/none reasoning effort) -- 30
theorems (a deterministic subset of main v3's 100, via the shared
`theorems.seed`) × 10 rungs × 4 model configs × 3 rollouts. The full-scale
cost projection (~$495) is well over the original $200 budget for this
sweep; see the config cell's comment block for cheaper subsets before
launching (e.g. drop `sonnet-4.6-thinking`, the single most expensive
config).


In [ ]:
RUN_DIR = results_root() / "runs" / MAIN_V3_2_CONFIG["run_name"]
sweep(MAIN_V3_2_CONFIG, RUN_DIR)


In [ ]:
write_run_analysis(RUN_DIR)
print((RUN_DIR / "analysis.txt").read_text())


## Noise Iso 3-way

Three-model isolation across the (MoE × reasoning) grid -- Mixtral 8x22B
(MoE, no reasoning), Nemotron-3-Super-120B (MoE, reasoning), Llama 3.3 70B
(dense, no reasoning) -- restricted to `hint:2..4` / `noise:3..4` (the rungs
where the noise-vs-hint marginal-content comparison is defined), 30 theorems
× 5 rungs × 3 models × 1 rollout.


In [ ]:
RUN_DIR = results_root() / "runs" / NOISE_ISO_3WAY_CONFIG["run_name"]
sweep(NOISE_ISO_3WAY_CONFIG, RUN_DIR)


In [ ]:
write_run_analysis(RUN_DIR)
print((RUN_DIR / "analysis.txt").read_text())


## Figures & CLI escape hatch

Publication figures are generated by the standalone scripts under
`notebooks/lean/figures/*.py`, run on the **main** `.venv` (they need
`matplotlib`, not `lean_dojo`):

```sh
.venv/bin/python notebooks/lean/figures/success_rate_bars.py --runs main_v3 main_v3_2
.venv/bin/python notebooks/lean/figures/success_rate_per_model_rung.py
.venv/bin/python notebooks/lean/figures/success_rate_with_noise.py
.venv/bin/python notebooks/lean/figures/response_length_per_model_rung.py
.venv/bin/python notebooks/lean/figures/marginal_content_vs_noise.py
.venv/bin/python notebooks/lean/figures/prompt_length_vs_hint.py
```

Every script accepts `--runs <run_name> [<run_name> ...]` (default varies by
script; see `smolbench.lean.figures.DEFAULT_RUNS`) and reads rows via
`smolbench.lean.figures.load_rows`, independent of this notebook's kernel.

For headless runs (e.g. a long sweep kicked off outside a notebook kernel,
or from a machine without a Jupyter setup), `run-sweep` still accepts a
YAML config path -- dump any of the dicts above with `yaml.safe_dump` if you
want a YAML copy of a notebook-defined sweep:

```sh
.venv-lean/bin/python -m smolbench.lean.cli run-sweep --config path/to/config.yaml
```

Other useful `smolbench.lean.cli` subcommands (see
`notebooks/lean/README.md` for the full list and which venv each needs):
`metadata`, `list`, `analyze`, `report`, `show`, `compare`, `prompt-stats`
(main `.venv`); `replay`, `filter`, `run-cell`, `run-sweep` (`.venv-lean`).
